# EmpathyTransformer V4 — Training on Kaggle (T4 x2)
Pilih salah satu setup: **A** (Kaggle Dataset) atau **B** (git clone). Jalanin salah satu.

## Setup A: Copy dari Kaggle Dataset (upload aitest/ dulu)

In [ ]:
import os, shutil
INPUT_DIR = '/kaggle/input/aitest'
WORK_DIR = '/kaggle/working/aitest'
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
shutil.copytree(INPUT_DIR, WORK_DIR)
os.chdir(WORK_DIR)
print('OK:', os.getcwd())

In [ ]:
!git pull
print('Git OK — kode terbaru')

## Setup B: Git Clone langsung dari repo (gak perlu upload dataset)

In [ ]:
import os, shutil
WORK_DIR = '/kaggle/working/aitest'
REPO_URL = 'https://github.com/BeatricKiersten/testsz.git'
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
!git clone $REPO_URL $WORK_DIR
os.chdir(WORK_DIR)
print('OK:', os.getcwd())

## Cek GPU

In [ ]:
import torch
print(f'GPUs: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}, {p.total_memory/1024**3:.1f}GB VRAM')

## Install dependencies

In [ ]:
!pip install tokenizers tqdm datasets -q
print('Deps OK')

## 1. Download Cosmopedia + Merge Dataset

In [ ]:
import os; os.chdir('/kaggle/working/aitest')
!python download_cosmopedia.py --output data/cosmopedia.jsonl --max-samples 30000 --subset 100k --merge data/train_mixed.jsonl data/train_empathy.jsonl data/conversations.jsonl --merge-output data/train_full.jsonl

In [ ]:
import os
for f in ['data/train_full.jsonl', 'data/cosmopedia.jsonl']:
    if os.path.exists(f):
        n = sum(1 for _ in open(f))
        sz = os.path.getsize(f) / 1024**2
        print(f'{os.path.basename(f)}: {n} lines, {sz:.1f} MB')

## 2. Train Tokenizer (skip kalo udah ada)

In [ ]:
import os; os.chdir('/kaggle/working/aitest')
DATA_PATH = 'data/train_full.jsonl'
TOK_PATH = 'tokenizer.json'
if os.path.exists(TOK_PATH):
    print(f'Tokenizer exists: {TOK_PATH} — delete to retrain')
else:
    !python tokenizer.py --data $DATA_PATH --vocab-size 16384 --save $TOK_PATH
    print('Tokenizer created')

## 3A. Train From Scratch

In [ ]:
import os; os.chdir('/kaggle/working/aitest')
!python train.py --data data/train_full.jsonl --tokenizer tokenizer.json --epochs 10 --batch-size 16 --grad-accum 4 --lr 3e-4 --warmup-steps 800 --dropout 0.2 --emb-lr-mult 1.0 --no-scale-emb --amp --checkpoint-dir ./checkpoints --log-every 10 --save-every 1000 --val-every 500

## 3B. Resume Training (kalo ada checkpoint)

In [ ]:
import os; os.chdir('/kaggle/working/aitest')
checkpoints = sorted([f for f in os.listdir('checkpoints') if f.endswith('.pt')]) if os.path.isdir('checkpoints') else []
if checkpoints:
    print('Available:')
    for c in checkpoints:
        sz = os.path.getsize(f'checkpoints/{c}')/1024**2
        print(f'  {c} ({sz:.1f} MB)')
else:
    print('No checkpoints — run 3A first')

In [ ]:
# Uncomment to resume:
# !python train.py --resume ./checkpoints/best.pt --tokenizer tokenizer.json --data data/train_full.jsonl --epochs 10 --batch-size 16 --grad-accum 4 --lr 5e-4 --amp

## 4. Test Inference

In [ ]:
%run inference.py --model checkpoints/best.pt --tokenizer tokenizer.json --prompt "I feel lonely today"

## 5. Save ke Kaggle Output

In [ ]:
import shutil, os
OUT = '/kaggle/working/aitest_checkpoints'
os.makedirs(OUT, exist_ok=True)
for f in ['best.pt', 'final.pt']:
    src = f'/kaggle/working/aitest/checkpoints/{f}'
    if os.path.exists(src):
        shutil.copy2(src, OUT)
        print(f'Copied {f}')
if os.path.exists('/kaggle/working/aitest/tokenizer.json'):
    shutil.copy2('/kaggle/working/aitest/tokenizer.json', OUT)
    print('Copied tokenizer.json')
for f in os.listdir(OUT):
    sz = os.path.getsize(os.path.join(OUT, f))/1024**2
    print(f'  {f}: {sz:.1f} MB')